# 🎤 YingMusic-SVC — Google Colab

Zero-shot Singing Voice Conversion: เปลี่ยนเสียงร้องในเพลงให้เป็นเสียงของคนอื่น (ใช้แค่ไฟล์ตัวอย่างเสียง 10–25 วินาที)

**ขั้นตอน**
1. `Runtime → Change runtime type → T4 GPU` (หรือ GPU ที่ดีกว่า)
2. รัน Cell **1 → 4** ตามลำดับ (ติดตั้ง + โหลดโมเดล ~5–8 นาที ครั้งแรก)
3. เลือกใช้งานแบบใดแบบหนึ่ง
   - **5. Gradio UI** — เปิดเว็บ UI ผ่านลิงก์ `*.gradio.live`
   - **6. Pipeline ใน Notebook** — อัปโหลดเพลง → แยกเสียงร้อง → แปลงเสียง → มิกซ์กลับ → ดาวน์โหลด

> ⚠️ ทุกขั้นที่ใช้ PyTorch รันเป็น subprocess (`!python ...`) จึง **ไม่ต้อง Restart runtime** หลังติดตั้ง แม้ pip จะเตือนเรื่อง version conflict ของแพ็กเกจอื่นใน Colab ก็ข้ามได้

In [ ]:
#@title 1. ตรวจสอบ GPU และ Python
import sys, subprocess
print("Python", sys.version.split()[0])
if sys.version_info[:2] > (3, 12):
    print("⚠️ Python > 3.12: torch 2.4.0 ไม่มี wheel ให้ — การติดตั้งใน Cell 3 อาจล้มเหลว")
r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("❌ ไม่พบ GPU — ไปที่ Runtime → Change runtime type → เลือก T4 GPU แล้วรันใหม่")
print(r.stdout)

In [ ]:
#@title 2. Clone repository
REPO_URL = "https://github.com/DDME36/YingMusic-SVC"  #@param {type:"string"}
BRANCH = ""  #@param {type:"string"}
#@markdown เว้น `BRANCH` ว่างไว้ = ใช้ default branch

import os
REPO_DIR = "/content/YingMusic-SVC"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    branch_flag = f"-b {BRANCH}" if BRANCH else ""
    !git clone --depth 1 $branch_flag $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git log -1 --oneline

In [ ]:
#@title 3. ติดตั้ง dependencies (~3–5 นาที)
%cd /content/YingMusic-SVC
# SoX จำเป็นสำหรับ torchaudio.sox_effects (echo/reverb ตอนมิกซ์กลับ)
!apt-get -qq update > /dev/null && apt-get -qq install -y sox libsox-dev libsox-fmt-all ffmpeg > /dev/null

# ล็อก torch ให้ตรงกับที่ repo ใช้ (torchaudio >= 2.9 ตัด sox_effects ออกแล้ว)
TORCH_PINS = "torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0"
!pip install -q $TORCH_PINS --index-url https://download.pytorch.org/whl/cu121
!pip install -q $TORCH_PINS --extra-index-url https://download.pytorch.org/whl/cu121 \
    "numpy==1.26.4" "scipy==1.13.1" "librosa==0.10.2" "transformers==4.46.3" \
    "huggingface-hub>=0.28.1,<1.0" "munch==4.0.0" "einops==0.8.0" "descript-audio-codec==1.0.0" \
    "gradio==5.23.0" "pydantic==2.10.6" "pydub==0.25.1" "soundfile==0.12.1" pyyaml \
    omegaconf ml_collections loralib wandb "beartype==0.14.1" "rotary_embedding_torch==0.3.5" matplotlib

!python -c "import torch, torchaudio, numpy; from torchaudio import sox_effects; print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| numpy', numpy.__version__, '| CUDA', torch.cuda.is_available())"

In [ ]:
#@title 4. ดาวน์โหลดโมเดล (~3 GB)
#@markdown ใส่ `HF_TOKEN` เฉพาะกรณีที่ Hugging Face ขอ login
HF_TOKEN = ""  #@param {type:"string"}

import os
os.chdir("/content/YingMusic-SVC")
from huggingface_hub import hf_hub_download, snapshot_download
token = HF_TOKEN or None

# โมเดลหลัก
SVC_CKPT = os.path.abspath(hf_hub_download("GiantAILab/YingMusic-SVC", "YingMusic-SVC-full.pt",
                                           local_dir="checkpoints", token=token))
SEP_CKPT = os.path.abspath(hf_hub_download("GiantAILab/YingMusic-SVC", "bs_roformer.ckpt",
                                           local_dir="accom_separation/ckpt/bs_roformer", token=token))
SEP_CFG = os.path.abspath("accom_separation/ckpt/bs_roformer/config_bd_roformer.yaml")

# โมเดลเสริม — โหลดไว้ล่วงหน้าใน cache path เดียวกับที่ my_inference.py / hf_utils.py ใช้
hf_hub_download("lj1995/VoiceConversionWebUI", "rmvpe.pt", cache_dir="./checkpoints")
hf_hub_download("funasr/campplus", "campplus_cn_common.bin", cache_dir="./checkpoints")
snapshot_download("nvidia/bigvgan_v2_44khz_128band_512x", cache_dir="./checkpoints/hf_cache",
                  allow_patterns=["config.json", "bigvgan_generator.pt"])
snapshot_download("openai/whisper-small", cache_dir="./checkpoints/hf_cache",
                  allow_patterns=["*.json", "*.safetensors", "*.txt"])

os.environ["YINGMUSIC_SVC_CKPT"] = SVC_CKPT
print("SVC checkpoint :", SVC_CKPT)
print("Separator ckpt :", SEP_CKPT)

---
## 5. Gradio UI
รัน Cell ด้านล่าง แล้วรอจนขึ้น `Running on public URL: https://xxxx.gradio.live` → คลิกเปิด

- ช่อง **SVC Checkpoint** ในแท็บ *Global Settings* จะกรอกให้อัตโนมัติ (ถ้าว่าง ให้วาง path ที่ Cell 4 พิมพ์ไว้)
- โมเดลจะโหลดครั้งแรกที่กด Convert (~1 นาที) ครั้งต่อไปจะเร็วขึ้น
- Cell นี้จะรันค้างไว้ตลอด — กด ⏹ เพื่อปิด UI ก่อนไปใช้ส่วนที่ 6

In [ ]:
#@title 5. เปิด Gradio UI
%cd /content/YingMusic-SVC
!PYTHONUNBUFFERED=1 python -u -c "import gradio_app as g; g.build_ui().queue().launch(share=True, server_name='0.0.0.0', server_port=7860)"

---
## 6. Pipeline ใน Notebook (ไม่ใช้ UI)

In [ ]:
#@title 6.1 เลือกไฟล์อินพุต
#@markdown **เพลงต้นฉบับ** — `upload`: อัปโหลดเพลงของคุณ, `sample`: ใช้เพลงตัวอย่างใน repo
SONG = "upload"  #@param ["upload", "sample"]
#@markdown ติ๊กถ้าเพลงมีดนตรีประกอบ (จะแยกเสียงร้องก่อน) — ถ้าเป็นไฟล์เสียงร้องล้วนให้เอาติ๊กออก
HAS_ACCOMPANIMENT = True  #@param {type:"boolean"}
#@markdown **เสียงเป้าหมาย (timbre)** — เสียงร้อง/พูดที่สะอาด ไม่มีดนตรี ยาว 10–25 วินาที
TIMBRE = "prompts/tp.wav"  #@param ["prompts/tp.wav", "prompts/syz.wav", "upload"]

import os, glob, shutil, time
from google.colab import files
os.chdir("/content/YingMusic-SVC")

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
WORK = os.path.abspath(f"colab_runs/{RUN_ID}")
os.makedirs(WORK, exist_ok=True)

def upload_one(prompt):
    print(prompt)
    up = files.upload()
    if not up:
        raise SystemExit("ไม่ได้อัปโหลดไฟล์")
    name = next(iter(up))
    dst = os.path.join(WORK, "upload_" + name.replace(" ", "_"))
    shutil.move(name, dst)
    return dst

if SONG == "upload":
    song_raw = upload_one("อัปโหลดเพลงต้นฉบับ (wav/mp3/flac/m4a):")
else:
    song_raw = glob.glob("accom_separation/samples/raw/*.MP3")[0]

# แปลงเป็น wav 44.1kHz (ชื่อไฟล์ไม่มีช่องว่าง)
SONG_WAV = os.path.join(WORK, "song.wav")
!ffmpeg -hide_banner -loglevel error -y -i "$song_raw" -ar 44100 "$SONG_WAV"

if TIMBRE == "upload":
    timbre_raw = upload_one("อัปโหลดเสียงเป้าหมาย (10–25 วินาที):")
    TIMBRE_WAV = os.path.join(WORK, "timbre.wav")
    !ffmpeg -hide_banner -loglevel error -y -i "$timbre_raw" -ar 44100 -ac 1 "$TIMBRE_WAV"
else:
    TIMBRE_WAV = os.path.abspath(TIMBRE)

print("Song  :", SONG_WAV)
print("Timbre:", TIMBRE_WAV)

In [ ]:
#@title 6.2 แยกเสียงร้อง / ดนตรี (BS-RoFormer)
#@markdown ติ๊กถ้าอยากเก็บเสียงคอรัส (backing vocal) ของต้นฉบับไว้ในดนตรีตอนมิกซ์กลับ
KEEP_BACKING_VOCAL = False  #@param {type:"boolean"}

import os
os.chdir("/content/YingMusic-SVC")
if HAS_ACCOMPANIMENT:
    SEP_IN, SEP_OUT = os.path.join(WORK, "sep_in"), os.path.join(WORK, "sep_out")
    os.makedirs(SEP_IN, exist_ok=True)
    shutil.copy(SONG_WAV, SEP_IN)
    !cd accom_separation && PYTHONWARNINGS=ignore python inference.py \
        --model_type bs_roformer --config_path "$SEP_CFG" --start_check_point "$SEP_CKPT" \
        --input_folder "$SEP_IN" --store_dir "$SEP_OUT" --disable_detailed_pbar
    stems = os.path.join(SEP_OUT, "song")
    SOURCE_VOCAL = os.path.join(stems, "vocals.wav")
    ACCOMPANY = os.path.join(stems, "instrumental.wav")
    if KEEP_BACKING_VOCAL:
        backing = os.path.join(stems, "backing_vocal.wav")
        ACCOMPANY = os.path.join(stems, "instrumental_with_backing.wav")
        !ffmpeg -hide_banner -loglevel error -y -i "{stems}/instrumental.wav" -i "$backing" \
            -filter_complex "amix=inputs=2:normalize=0" "$ACCOMPANY"
    for p in (SOURCE_VOCAL, ACCOMPANY):
        assert os.path.isfile(p), f"ไม่พบไฟล์ {p} — ดู log ด้านบน"
else:
    SOURCE_VOCAL, ACCOMPANY = SONG_WAV, ""

print("Source vocal :", SOURCE_VOCAL)
print("Accompaniment:", ACCOMPANY or "(ไม่มี)")

In [ ]:
#@title 6.3 แปลงเสียง (SVC)
#@markdown `DIFFUSION_STEPS` มาก = คุณภาพนิ่งขึ้นแต่ช้าลง (100 = ค่าที่ผู้พัฒนาแนะนำ, 30–50 = เร็ว)
DIFFUSION_STEPS = 100  #@param {type:"slider", min:10, max:200, step:10}
FP16 = True  #@param {type:"boolean"}

import os, glob
os.chdir("/content/YingMusic-SVC")
EXP = f"colab_{RUN_ID}"
!python -u my_inference.py \
    --source "$SOURCE_VOCAL" --target "$TIMBRE_WAV" \
    --diffusion-steps $DIFFUSION_STEPS --checkpoint "$SVC_CKPT" \
    --expname "$EXP" --cuda 0 --fp16 $FP16 \
    --accompany "$ACCOMPANY" --config configs/YingMusic-SVC.yml

out_dir = os.path.join("outputs", EXP)
VC_FILES = sorted(glob.glob(os.path.join(out_dir, "*.wav")))
MIX_FILES = sorted(glob.glob(os.path.join(out_dir, "accompany", "*.wav")))
assert VC_FILES, "ไม่พบไฟล์ผลลัพธ์ — ดู error ด้านบน"
print("Converted vocal:", VC_FILES[-1])
print("Mixed song     :", MIX_FILES[-1] if MIX_FILES else "(ไม่มี)")

In [ ]:
#@title 6.4 ฟัง & ดาวน์โหลด
DOWNLOAD = True  #@param {type:"boolean"}
import os
from IPython.display import Audio, display
from google.colab import files

results = [("เสียงที่แปลงแล้ว", VC_FILES[-1])]
if MIX_FILES:
    results.append(("มิกซ์กับดนตรี", MIX_FILES[-1]))

for label, wav in results:
    # preview เป็น mp3 เพื่อไม่ให้ notebook หนัก
    mp3 = os.path.splitext(wav)[0] + ".mp3"
    !ffmpeg -hide_banner -loglevel error -y -i "$wav" -b:a 192k "$mp3"
    print(label, "→", wav)
    display(Audio(filename=mp3))
    if DOWNLOAD:
        files.download(wav)